# Fitness App User Segmentation

This notebook performs an end-to-end segmentation of fitness app users.

Contents:
- Data generation (creates `fitness_users.csv` with 500 rows) if file not present
- EDA: distributions, correlations, missing-values/missingness diagnostics
- Preprocessing: imputation and outlier handling
- K-Means clustering (choose K using Elbow + Silhouette) for K in 3..5
- Visualizations: pairplots, PCA-based 2D, interactive 3D Plotly, parallel coordinates
- Cluster profiling and business-oriented recommendations

Unique touches:
- 3D interactive visualization
- Parallel coordinates to highlight multi-dimensional tradeoffs
- Automated short recommendations per cluster and a sample-user cluster predictor

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')


## 1) Generate dataset if not present
We include a generator so you can reproduce data and tweak distributions. If you already have `fitness_users.csv` it will be loaded instead.

In [ ]:
if not os.path.exists('fitness_users.csv'):
    print('fitness_users.csv not found — generating synthetic dataset...')
    # Generate as in generate_dataset.py logic
    import random
    random.seed(42)
    rows = []
    n = 500
    wk = ['Cardio','Strength','Flexibility']
    for i in range(1, n+1):
        uid = f'U{i:04d}'
        age = 18 + ((i * 13) % 43)
        gender = 'Male' if ((i * 7) % 2 == 0) else 'Female'
        wf = (i * 3) % 8
        if wf >= 4:
            pref = random.choices(wk, weights=(0.5,0.3,0.2))[0]
        else:
            pref = random.choices(wk, weights=(0.4,0.35,0.25))[0]
        base_steps = 3000 + wf * 1200 + ((i * 131) % 8000)
        age_penalty = int((age - 30) * 30) if age > 30 else 0
        steps = max(800, min(20000, int(base_steps - age_penalty + random.gauss(0, 800))))
        cal_from_steps = steps * 0.04
        base_cal = 1700 + cal_from_steps + wf * 30 + random.gauss(0, 120)
        calories = int(max(1400, min(3800, base_cal)))
        sleep = max(4.0, min(9.5, round(5.0 + ((i * 17) % 45) / 10.0 + random.gauss(0,0.6), 1)))
        # Introduce missing values intentionally
        calories_field = '' if i % 37 == 0 else str(calories)
        sleep_field = '' if i % 53 == 0 else str(sleep)
        steps_field = '' if i % 41 == 0 else str(steps)
        rows.append([uid, age, gender, steps_field, calories_field, wf, pref, sleep_field])
    df_gen = pd.DataFrame(rows, columns=['UserID','Age','Gender','AvgDailySteps','AvgDailyCalories','WorkoutFrequencyPerWeek','PreferredWorkoutType','SleepHours'])
    df_gen.to_csv('fitness_users.csv', index=False)
    print('Saved fitness_users.csv (500 rows)')
else:
    print('fitness_users.csv found — loading it')


In [ ]:
df = pd.read_csv('fitness_users.csv')
df.head()

### Quick summary and missing values

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())
display(df.describe(include='all').T)


Convert numeric columns to numeric types and inspect distributions.

In [ ]:
for col in ['AvgDailySteps','AvgDailyCalories','SleepHours']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df[['AvgDailySteps','AvgDailyCalories','SleepHours']].describe().round(2)

## EDA: Distributions & pairwise relationships

In [ ]:
import matplotlib.ticker as mtick
fig, axes = plt.subplots(2,3, figsize=(15,8))
sns.histplot(df['Age'].dropna(), bins=20, ax=axes[0,0], color='skyblue').set_title('Age')
sns.histplot(df['AvgDailySteps'].dropna(), bins=30, ax=axes[0,1], color='salmon').set_title('AvgDailySteps')
sns.histplot(df['AvgDailyCalories'].dropna(), bins=30, ax=axes[0,2], color='lightgreen').set_title('AvgDailyCalories')
sns.countplot(y='PreferredWorkoutType', data=df, ax=axes[1,0], palette='pastel').set_title('PreferredWorkoutType')
sns.histplot(df['WorkoutFrequencyPerWeek'].dropna(), bins=8, ax=axes[1,1], color='violet').set_title('WorkoutFrequencyPerWeek')
sns.histplot(df['SleepHours'].dropna(), bins=15, ax=axes[1,2], color='orange').set_title('SleepHours')
plt.tight_layout()


Pairplot (subsample for performance)

In [ ]:
sample = df.sample(300, random_state=42)
sns.pairplot(sample[['Age','AvgDailySteps','AvgDailyCalories','WorkoutFrequencyPerWeek','SleepHours']].dropna().reset_index(drop=True))


### Missingness analysis

In [ ]:
missing = df.isnull().mean().round(3) * 100
missing[missing>0]


## Handling missing values and outliers
- We will impute numeric missing values with median.
- For outliers we use IQR-based capping (winsorization) to reduce undue K-Means influence.

In [ ]:
df_proc = df.copy()
for col in ['AvgDailySteps','AvgDailyCalories','SleepHours']:
    med = df_proc[col].median()
    df_proc[col] = df_proc[col].fillna(med)

def cap_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    return series.clip(lo, hi)

for col in ['AvgDailySteps','AvgDailyCalories','SleepHours']:
    df_proc[col] = cap_iqr(df_proc[col])

df_proc[['AvgDailySteps','AvgDailyCalories','SleepHours']].describe().round(2)


## Feature selection, scaling and clustering
We'll try K from 3 to 5 and present inertia and silhouette scores to guide selection.

In [ ]:
features = ['Age','AvgDailySteps','AvgDailyCalories','WorkoutFrequencyPerWeek','SleepHours']
X = df_proc[features].values
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

inertias = {}
sil = {}
for k in range(3,6):
    km = KMeans(n_clusters=k, random_state=42, n_init=30)
    labels = km.fit_predict(Xs)
    inertias[k] = km.inertia_
    sil[k] = silhouette_score(Xs, labels)

inertias, sil


In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(list(inertias.keys()), list(inertias.values()), marker='o')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.title('Elbow: Inertia vs K')
plt.subplot(1,2,2)
plt.plot(list(sil.keys()), list(sil.values()), marker='o', color='orange')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.title('Silhouette vs K')
plt.tight_layout()


Based on the plots and silhouette scores choose K. For this dataset we will pick K=4 for demonstration (balanced interpretability & separation).

In [ ]:
chosen_k = 4
kmeans = KMeans(n_clusters=chosen_k, random_state=42, n_init=50)
labels = kmeans.fit_predict(Xs)
df_proc['cluster'] = labels
df_proc['cluster'] = df_proc['cluster'].astype(int)
df_proc['cluster'].value_counts().sort_index()


### Visualize clusters (2D PCA projection)

In [ ]:
pca = PCA(n_components=2, random_state=42)
pcs = pca.fit_transform(Xs)
df_proc['pca1'] = pcs[:,0]
df_proc['pca2'] = pcs[:,1]
plt.figure(figsize=(8,6))
sns.scatterplot(data=df_proc, x='pca1', y='pca2', hue='cluster', palette='tab10', alpha=0.8)
plt.title('PCA projection colored by cluster')
plt.show()


### Interactive 3D cluster plot (Plotly)

In [ ]:
pca3 = PCA(n_components=3, random_state=42)
pcs3 = pca3.fit_transform(Xs)
df_proc['pca3_1'] = pcs3[:,0]
df_proc['pca3_2'] = pcs3[:,1]
df_proc['pca3_3'] = pcs3[:,2]
fig = px.scatter_3d(df_proc, x='pca3_1', y='pca3_2', z='pca3_3', color='cluster', hover_data=['UserID','Age','Gender','PreferredWorkoutType'], title='3D PCA cluster view')
fig.show()


### Pairplot colored by cluster (use a subsample for readability)

In [ ]:
sns.pairplot(df_proc.sample(300, random_state=42), vars=['Age','AvgDailySteps','AvgDailyCalories','WorkoutFrequencyPerWeek','SleepHours'], hue='cluster', palette='tab10', plot_kws={'alpha':0.7, 's':40})


### Parallel coordinates to inspect multi-feature tradeoffs

In [ ]:
from pandas.plotting import parallel_coordinates
pc_sample = df_proc.sample(250, random_state=42)
plt.figure(figsize=(12,6))
parallel_coordinates(pc_sample[['cluster','Age','AvgDailySteps','AvgDailyCalories','WorkoutFrequencyPerWeek','SleepHours']], 'cluster', colormap='tab10', alpha=0.6)
plt.title('Parallel coordinates (subsample)')
plt.xticks(rotation=45)
plt.show()


## Cluster profiling and interpretation
Compute means and sizes; then craft business-friendly descriptions and recommended actions for each cluster.

In [ ]:
profile = df_proc.groupby('cluster')[['Age','AvgDailySteps','AvgDailyCalories','WorkoutFrequencyPerWeek','SleepHours']].agg(['mean','median','count']).round(2)
profile

### Human-friendly interpretation (examples)
We generate concise interpretations using simple heuristics to highlight how product/marketing could act on each segment.

In [ ]:
profiles = df_proc.groupby('cluster')[['Age','AvgDailySteps','AvgDailyCalories','WorkoutFrequencyPerWeek','SleepHours']].mean()
median_steps = df_proc['AvgDailySteps'].median()
median_cal = df_proc['AvgDailyCalories'].median()
median_wf = df_proc['WorkoutFrequencyPerWeek'].median()
median_sleep = df_proc['SleepHours'].median()

interpretations = {}
for cl in profiles.index:
    r = profiles.loc[cl]
    desc = []
    if r['AvgDailySteps'] > median_steps and r['WorkoutFrequencyPerWeek'] >= median_wf:
        desc.append('Active')
    if r['AvgDailySteps'] < median_steps and r['WorkoutFrequencyPerWeek'] <= median_wf:
        desc.append('Low-activity')
    if r['AvgDailyCalories'] > median_cal:
        desc.append('Higher calorie burn')
    if r['SleepHours'] < median_sleep:
        desc.append('Lower sleep')
    label = ', '.join(desc) if desc else 'Moderate/Regular'
    # recommendations
    reco = []
    if 'Active' in label:
        reco.append('Promote advanced challenges and premium plan trials.')
    if 'Low-activity' in label:
        reco.append('Send starter programs, short-term nudges and streak incentives.')
    if 'Lower sleep' in label:
        reco.append('Recommend sleep and recovery content; remind about rest days.')
    if not reco:
        reco = ['Maintain engagement with weekly content suggestions.']
    interpretations[int(cl)] = {'label': label, 'recommendations': reco}

for cl, info in interpretations.items():
    print(f"Cluster {cl}: {info['label']}")
    for r in info['recommendations']:
        print('  -', r)
    print()


## Save clustered dataset for downstream use

In [ ]:
df_proc.to_csv('fitness_users_clustered.csv', index=False)
print('Saved fitness_users_clustered.csv')


## Next steps / Extensions
- Use time-series features (e.g., weekly step trends) to find behavioral dynamics.
- Build a recommender system that maps each cluster to a sequence of in-app experiences A/B-tested for conversion.
- Consider other clustering methods (e.g., Gaussian Mixture Models, HDBSCAN) and compare with silhouette and stability metrics.
- Deploy the Streamlit dashboard from this project as a product tool for marketing/product teams.
